# Rappel de l objectif rendefinssant les euristic , les meta heur, diff entre diversification et intensification.

## Contexte



## Modeling and solution methods : 

### Data formalization : 
The Graph: The network is a complete directed graph $G = (V, A)$ where $V$ is the set of nodes (the depot $0$ plus the customers $C$). 
<br>
The Fleets: A set of homogeneous vehicles $K$, each with a maximum capacity $Q$. 
<br>
The Customers: Each customer $i \in C$ has a demand $q_i$, a service time $s_i$, and a strict time window $[e_i, l_i]$ (start and end times). 
<br>
Edges: Each edge $(i, j) \in A$ has a travel cost $c_{ij}$ (distance) and a travel time $t_{ij}$.

### Complexity proof : 
The CVRPTW is a generalization of the TSP (Traveling Salesman Problem) and the Bin Packing Problem. Since the TSP is NP-hard, the CVRPTW is logically NP-hard as well.<br>
The combinatorial explosion therefore makes exact methods impractical.

A VERIFIER

### Complete and corrected mathematical modeling in the form of an Integer Linear Program (ILP),

### A presentation of the potential applications of this exact approach (direct solution, benchmark, lower/upper bounds, etc.) (ALEXANDRE)



### Difference between Diversification and Intensification

## Data set cf. GitHub : https://github.com/PyVRP/VRPLIB

In [39]:
#import vrplib

# Remplacez par le vrai nom d'un de vos fichiers (ex: data/C101.txt)
#instance = vrplib.read_instance("data/C101.txt", instance_format="solomon") 
#solution_opt = vrplib.read_solution("data/C101.sol")

# Afficher ce que contient l'instance
#print(instance.keys())
#print("Score optimal à battre :", solution_opt.get("cost", solution_opt.get("Cost")))

## Construction du Graph 


In [51]:
import random
from collections import deque

# =======================================================
# 0. STRUCTURE DE DONNÉES (Classes)
# =======================================================

class Node:
    """Modélise un sommet du réseau (Dépôt ou Client) avec ses paramètres."""
    def __init__(self, node_id, demand=0, earliest_time=0, latest_time=float('inf'), service_time=0):
        self.node_id = node_id  # Indice i
        self.q = demand         # Demande q_i
        self.e = earliest_time  # Ouverture de la fenêtre e_i
        self.l = latest_time    # Fermeture de la fenêtre l_i
        self.s = service_time   # Temps de service s_i

class CVRPTW_Network:
    """Modélise le graphe orienté complet G = (V, E) et les paramètres du CVRPTW."""
    def __init__(self, n_customers, k_trucks, capacity):
        self.N = n_customers    # Nombre total de clients N
        self.K = k_trucks       # Nombre de camions K
        self.Q = capacity       # Capacité maximale d'un camion Q
        self.B = float('inf')   # Borne de coût B'

        self.V = list(range(self.N + 1))
        self.C = [node for node in self.V if node != 0]
        self.E = [(i, j) for i in self.V for j in self.V if i != j]

        self.nodes = {}
        self.travel_times = {}

    def add_node(self, node_id, demand=0, e=0, l=float('inf'), s=0):
        self.nodes[node_id] = Node(node_id, demand, e, l, s)

    def set_travel_time(self, i, j, time_val):
        if (i, j) in self.E:
            self.travel_times[(i, j)] = time_val

    def calculate_objective(self, routes):
        """Calcule la fonction objectif Z = Somme(C_ij * X_ij)."""
        total_time = 0
        for route in routes:
            for idx in range(len(route) - 1):
                i = route[idx]
                j = route[idx + 1]
                total_time += self.travel_times.get((i, j), 0)
        return total_time




In [ ]:
def charger_instance_cvrptw(chemin_fichier, limite_clients=None):
    import vrplib
    
    # ⚠️ CORRECTION : Il FAUT préciser "solomon" pour lire les fichiers .txt
    data = vrplib.read_instance(chemin_fichier, instance_format="solomon") 
    print(data)
    # On réduit la taille du problème si une limite est donnée
    dimension_totale = len(data['demand'])
    if limite_clients is not None:
        dimension = min(limite_clients + 1, dimension_totale)
    else:
        dimension = dimension_totale
        
    n_clients = dimension - 1 
    capacite = data['capacity']
    nb_camions = data.get('vehicles', 25) 
    
    # Création du réseau
    reseau = CVRPTW_Network(n_customers=n_clients, k_trucks=nb_camions, capacity=capacite)
    
    for i in range(dimension):
        reseau.add_node(
            node_id=i,
            demand=data['demand'][i],
            e=data['time_window'][i][0],
            l=data['time_window'][i][1],
            s=data['service_time'][i]
        )
        
    distances = data['edge_weight']
    for i in reseau.V:
        for j in reseau.V:
            if i != j:
                reseau.set_travel_time(i, j, distances[i][j])
                
    return reseau

## Verificaton des fenetres de temps

In [53]:
def est_realisable(route_tuple, reseau):
    """Vérifie si une route respecte les capacités et fenêtres de temps."""
    charge_actuelle = 0
    temps_actuel = 0
    
    for k in range(len(route_tuple) - 1):
        i = route_tuple[k]
        j = route_tuple[k+1]
        
        charge_actuelle += reseau.nodes[i].q
        if charge_actuelle > reseau.Q:
            return False
            
        temps_actuel = max(temps_actuel, reseau.nodes[i].e)
        temps_actuel += reseau.nodes[i].s + reseau.travel_times.get((i, j), float('inf'))
        
        if temps_actuel > reseau.nodes[j].l:
            return False 
            
    return True

In [54]:
def solution_est_realisable(solution, reseau):
    """Vérificateur global : vérifie si TOUTES les routes de la solution sont valides."""
    for route in solution:
        if not est_realisable(route, reseau):
            return False
    return True

## Choix de la méthode de Voisinage

In [55]:
# =======================================================
# LES 3 OPÉRATEURS DE RECHERCHE LOCALE (MULTI-ROUTES)
# =======================================================

def voisinage_multi_swap(solution):
    """
    Méthode 1 : SWAP (Échange)
    Intervertit deux clients. Ils peuvent être dans la même route ou dans deux camions différents.
    """
    solution_list = [list(r) for r in solution]
    nb_routes = len(solution_list)
    
    for r1 in range(nb_routes):
        for i in range(1, len(solution_list[r1]) - 1): # Ignore les dépôts (0)
            for r2 in range(r1, nb_routes):
                # Pour ne pas refaire les mêmes échanges en double
                start_j = i + 1 if r1 == r2 else 1
                for j in range(start_j, len(solution_list[r2]) - 1):
                    voisin = [list(r) for r in solution_list]
                    # Échange des deux clients
                    voisin[r1][i], voisin[r2][j] = voisin[r2][j], voisin[r1][i]
                    
                    yield tuple(tuple(r) for r in voisin)

import random

def voisinage_multi_relocate(solution, max_voisins=150):
    """
    Version FAST (Stochastique) : tire un échantillon aléatoire de voisins 
    pour éviter l'explosion du temps de calcul (passe de 8 minutes à 10 secondes).
    """
    solution_list = [list(r) for r in solution]
    nb_routes = len(solution_list)
    voisins_generes = 0
    essais = 0
    
    # On limite la recherche à 'max_voisins' pour être ultra-rapide
    while voisins_generes < max_voisins and essais < max_voisins * 10:
        essais += 1
        
        # Tirage d'un client au hasard
        r1 = random.randint(0, nb_routes - 1)
        if len(solution_list[r1]) <= 2: continue # On ignore les camions vides
        i = random.randint(1, len(solution_list[r1]) - 2)
        client = solution_list[r1][i]
        
        # Tirage d'une destination au hasard
        r2 = random.randint(0, nb_routes - 1)
        j = random.randint(1, len(solution_list[r2]) - 1)
        
        if r1 == r2 and i == j: continue
            
        # Création du voisin
        voisin = [list(r) for r in solution_list]
        voisin[r1].pop(i)
        
        if r1 == r2 and j > i:
            voisin[r2].insert(j-1, client)
        else:
            voisin[r2].insert(j, client)
            
        voisin_propre = tuple(tuple(r) for r in voisin if len(r) > 2)
        
        voisins_generes += 1
        yield voisin_propre


def voisinage_multi_2opt(solution):
    """
    Méthode 3 : 2-OPT (Inversion de segment intra-route)
    Inverse totalement l'ordre de passage d'un sous-groupe de clients à l'intérieur d'un camion.
    """
    solution_list = [list(r) for r in solution]
    nb_routes = len(solution_list)
    
    for r in range(nb_routes):
        route = solution_list[r]
        n = len(route)
        for i in range(1, n - 2):
            for j in range(i + 1, n - 1):
                voisin = [list(rt) for rt in solution_list]
                # Inversion du segment [i:j]
                voisin[r][i:j+1] = reversed(voisin[r][i:j+1])
                
                yield tuple(tuple(rt) for rt in voisin)


# =======================================================
# LE SÉLECTEUR DE MÉTHODE UNIFIÉ
# =======================================================

def voisinage_multi(solution, methode="relocate"):
    """Sélecteur global pour générer le voisinage selon la méthode choisie."""
    if methode == "swap":
        return voisinage_multi_swap(solution)
    elif methode == "relocate":
        return voisinage_multi_relocate(solution)
    elif methode == "2opt":
        return voisinage_multi_2opt(solution)
    else:
        raise ValueError(f"Méthode '{methode}' inconnue. Choisissez 'swap', 'relocate' ou '2opt'.")

## Multi Start / GARSP ? 
Explication du principe et on l implémente pour chaque algo(c est une hypothèse)

In [56]:
def random_solution_multi(reseau):
    """Génère une solution initiale valide avec PLUSIEURS camions (Glouton aléatoire)."""
    clients_restants = set(reseau.C)
    routes = []
    
    while clients_restants:
        route_actuelle = [0]
        candidats = list(clients_restants)
        random.shuffle(candidats)
        
        # On remplit le camion tant que la capacité et le temps le permettent
        for client in candidats:
            test_route = route_actuelle + [client, 0]
            # On réutilise votre fonction 'est_realisable' qui marche très bien pour une route
            if est_realisable(tuple(test_route), reseau):
                route_actuelle.append(client)
                clients_restants.remove(client)
                
        # Le camion rentre au dépôt
        route_actuelle.append(0)
        routes.append(tuple(route_actuelle))
        
    return tuple(routes) # Tuple de tuples pour être immuable (requis pour la liste Tabou)


def lanceur_multi_start(reseau, algorithme, nb_demarrages=5, **kwargs_algo):
    """
    Exécute un algorithme d'optimisation plusieurs fois avec des points de départ différents.
    
    - reseau : Votre instance CVRPTW_Network
    - algorithme : Le nom de la fonction (ex: hill_climbing_multi, recherche_tabou_multi)
    - nb_demarrages : Nombre de fois où l'on recommence de zéro
    - **kwargs_algo : Les paramètres spécifiques à envoyer à l'algorithme (ex: methode_voisinage="swap")
    """
    meilleur_score_global = float('inf')
    meilleure_solution_globale = None
    
    print(f"🚀 Lancement Multi-Start ({nb_demarrages} démarrages) pour {algorithme.__name__}")
    
    for essai in range(1, nb_demarrages + 1):
        # 1. On génère un nouveau point de départ aléatoire
        sol_init = random_solution_multi(reseau)
        
        # 2. On lance l'algorithme demandé (en lui passant ses paramètres spécifiques)
        sol_optimisee = algorithme(sol_init, reseau, **kwargs_algo)
        
        # 3. On évalue le résultat
        score = reseau.calculate_objective(sol_optimisee)
        print(f"  ↳ Essai {essai}/{nb_demarrages} | Score: {score:.2f} | Camions: {len(sol_optimisee)}")
        
        # 4. On sauvegarde si c'est un nouveau record global
        if score < meilleur_score_global:
            meilleur_score_global = score
            meilleure_solution_globale = sol_optimisee
            print(f"      🌟 Nouveau Record Global : {meilleur_score_global:.2f}")
            
    return meilleure_solution_globale

In [57]:
def nearest_neighbor_solution_multi(reseau):
    """
    Heuristique Constructive (Multi-Routes) : Plus Proche Voisin.
    Génère une flotte de camions. Dès qu'un camion est plein ou hors délai, 
    il rentre au dépôt et un nouveau camion prend le relais.
    """
    clients_non_visites = set(reseau.C)
    routes = []
    
    # Tant qu'il reste des clients à livrer
    while clients_non_visites:
        route_courante = [0] # Nouveau camion part du dépôt
        temps_actuel = 0
        charge_actuelle = 0
        camion_peut_continuer = True
        
        while camion_peut_continuer and clients_non_visites:
            client_actuel = route_courante[-1]
            meilleur_prochain = None
            meilleur_cout = float('inf')
            meilleur_temps_arrivee = 0
            
            for candidat in clients_non_visites:
                # 1. Vérification de la Capacité Q
                if charge_actuelle + reseau.nodes[candidat].q > reseau.Q:
                    continue
                    
                t_trajet = reseau.travel_times.get((client_actuel, candidat), float('inf'))
                t_arrivee = temps_actuel + t_trajet
                
                # 2. Gestion de l'heure d'arrivée (attente si en avance)
                if t_arrivee < reseau.nodes[candidat].e:
                    t_arrivee = reseau.nodes[candidat].e
                    
                # 3. Vérification de la Fenêtre de Temps
                if t_arrivee <= reseau.nodes[candidat].l:
                    cout = t_arrivee - temps_actuel 
                    if cout < meilleur_cout:
                        meilleur_cout = cout
                        meilleur_prochain = candidat
                        meilleur_temps_arrivee = t_arrivee
            
            # Ajout du client trouvé
            if meilleur_prochain is not None:
                route_courante.append(meilleur_prochain)
                clients_non_visites.remove(meilleur_prochain)
                temps_actuel = meilleur_temps_arrivee + reseau.nodes[meilleur_prochain].s
                charge_actuelle += reseau.nodes[meilleur_prochain].q
            else:
                # Aucun client ne peut être ajouté à ce camion
                camion_peut_continuer = False 
                
        # Le camion rentre au dépôt
        route_courante.append(0)
        
        # Sécurité : Si un camion part et revient à vide (0, 0) alors qu'il reste des clients
        if len(route_courante) == 2 and clients_non_visites:
            raise Exception(f"Impossible de livrer les clients restants : {clients_non_visites} (Contraintes trop strictes)")
            
        routes.append(tuple(route_courante))
        
    return routes

## Heurisitcs 

### VND

### Hills Climbings
The current node is initialized to its initial state. Iteratively, the current node is compared to its immediate successors (neighbors). <br>The best immediate successor—that is, the one with the highest value compared to the current node—becomes the new current node. If no such neighbor exists, the process stops, and the current node is returned as the solution.

In [58]:
def hill_climbing_multi(solution_initiale, reseau, methode_voisinage="relocate"):
    solution_courante = solution_initiale
    nouveau = True
    
    while nouveau:
        meilleure_solution = solution_courante
        meilleur_score = reseau.calculate_objective(solution_courante)
        
        # On utilise ici l'argument methode_voisinage
        for voisin in voisinage_multi(solution_courante, methode=methode_voisinage):
            if solution_est_realisable(voisin, reseau):
                score_voisin = reseau.calculate_objective(voisin)
                if score_voisin < meilleur_score:
                    meilleure_solution = voisin
                    meilleur_score = score_voisin
                    
        nouveau = (meilleure_solution != solution_courante)
        solution_courante = meilleure_solution
        
    return solution_courante

In [59]:
def vnd_multi(solution_initiale, reseau):
    """
    Descente à Voisinage Variable (VND).
    Combine vos 3 opérateurs : si l'un bloque, il passe au suivant pour s'échapper.
    Ordre typique : 1. Relocate (précis), 2. Swap (intermédiaire), 3. 2-opt (lourd).
    """
    # L'ordre d'exploration de vos voisinages
    methodes = ["relocate", "swap", "2opt"]
    
    solution_courante = solution_initiale
    meilleur_score = reseau.calculate_objective(solution_courante)
    
    k = 0 # Index de la méthode actuelle
    
    while k < len(methodes):
        methode_actuelle = methodes[k]
        
        # On lance un Hill Climbing complet jusqu'à être bloqué avec CETTE méthode
        sol_hc = hill_climbing_multi(solution_courante, reseau, methode_voisinage=methode_actuelle)
        score_hc = reseau.calculate_objective(sol_hc)
        
        if score_hc < meilleur_score:
            # L'algorithme a réussi à améliorer le score !
            solution_courante = sol_hc
            meilleur_score = score_hc
            
            # Règle d'or du VND : dès qu'on s'améliore, on retourne à la méthode la plus fine
            k = 0 
        else:
            # Bloqué ! La méthode actuelle ne donne plus rien.
            # On passe à la méthode suivante (ex: on lâche le relocate pour tenter un swap)
            k += 1 
            
    return solution_courante

## Meta - Heurisitcs

### Tabu Search

In [60]:
from collections import deque

def recherche_tabou_multi(solution_initiale, reseau, taille_tabou, iter_max, methode_voisinage="relocate"):
    """
    Métaheuristique Tabou compatible avec plusieurs camions et les Datasets VRPLIB.
    """
    liste_tabou = deque(maxlen=taille_tabou)
    solution_courante = solution_initiale
    meilleure_globale = solution_initiale
    
    # APPEL CORRIGÉ : On passe la solution directement
    valeur_meilleure_globale = reseau.calculate_objective(solution_initiale)
    nb_iter_sans_amelioration = 0
    
    while nb_iter_sans_amelioration < iter_max:
        meilleur_voisin = None
        valeur_meilleur_voisin = float('inf')
        
        # APPEL CORRIGÉ : On utilise le sélecteur multi-voisinages
        for voisin in voisinage_multi(solution_courante, methode=methode_voisinage):
            
            # APPEL CORRIGÉ : On vérifie TOUS les camions avec solution_est_realisable
            if voisin not in liste_tabou and solution_est_realisable(voisin, reseau):
                score_voisin = reseau.calculate_objective(voisin)
                
                if score_voisin < valeur_meilleur_voisin:
                    meilleur_voisin = voisin
                    valeur_meilleur_voisin = score_voisin
        
        # Si aucun voisin n'est valide (cas très rare mais possible si très contraint)
        if meilleur_voisin is None:
            break
            
        # Mise à jour des records
        if valeur_meilleur_voisin < valeur_meilleure_globale:
            meilleure_globale = meilleur_voisin
            valeur_meilleure_globale = valeur_meilleur_voisin
            nb_iter_sans_amelioration = 0
        else:
            nb_iter_sans_amelioration += 1
            
        # On avance
        solution_courante = meilleur_voisin
        liste_tabou.append(solution_courante)
        
    return meilleure_globale

### Simulated Annealing

In [61]:
import math
import random

def recuit_simule_multi(solution_initiale, reseau, temp_initiale, taux_refroidissement, iter_max, methode_voisinage="relocate"):
    """
    Métaheuristique : Recuit Simulé (Simulated Annealing) compatible avec plusieurs camions.
    Accepte des détours selon une probabilité qui baisse avec le temps.
    """
    solution_courante = solution_initiale
    meilleure_globale = solution_initiale
    
    # APPEL CORRIGÉ : On calcule le coût en passant directement le tuple de routes
    valeur_courante = reseau.calculate_objective(solution_courante)
    valeur_meilleure_globale = valeur_courante
    
    temperature = temp_initiale
    
    for _ in range(iter_max):
        # APPEL CORRIGÉ : Voisinage multi-routes et vérification globale de la validité
        voisins_valides = [
            v for v in voisinage_multi(solution_courante, methode=methode_voisinage) 
            if solution_est_realisable(v, reseau)
        ]
        
        if not voisins_valides:
            break # Sécurité : aucun voisin possible sans violer les contraintes (poids, temps)
            
        # 1. On tire UN SEUL voisin au hasard
        voisin_choisi = random.choice(voisins_valides)
        valeur_voisin = reseau.calculate_objective(voisin_choisi)
        
        # 2. Calcul de la différence de score
        # delta < 0 : le voisin est plus rapide (amélioration)
        # delta > 0 : le voisin est plus lent (dégradation)
        delta = valeur_voisin - valeur_courante
        
        # 3. Critère d'acceptation (Metropolis)
        if delta < 0:
            # C'est meilleur : on accepte d'office l'amélioration
            solution_courante = voisin_choisi
            valeur_courante = valeur_voisin
        else:
            # C'est pire : on calcule la probabilité d'accepter ce détour
            if temperature > 0.001:
                # Formule mathématique d'acceptation
                probabilite = math.exp(-delta / temperature)
                
                # Le coup de dés
                if random.random() < probabilite:
                    solution_courante = voisin_choisi
                    valeur_courante = valeur_voisin
                    
        # 4. Mise à jour du record absolu
        if valeur_courante < valeur_meilleure_globale:
            meilleure_globale = solution_courante
            valeur_meilleure_globale = valeur_courante
            
        # 5. On refroidit la température
        temperature *= taux_refroidissement
        
    return meilleure_globale

## Test et print console

In [ ]:
import vrplib

# Configuration de l'instance C101
fichier_instance = "data/C101.txt"
fichier_solution = "data/C101.sol"

# Chargement (en supposant que vos fonctions de chargement sont définies plus haut)
reseau_c101 = charger_instance_cvrptw(fichier_instance)
sol_opt = vrplib.read_solution(fichier_solution)
score_optimal = sol_opt.get("cost", sol_opt.get("Cost"))
#score_optimal = 2.5
nb_camions_optimal = len(sol_opt.get("routes", []))

print("=========================================================")
print(f" BENCHMARK ADEME : Instance {fichier_instance}")
print("=========================================================\n")

try:
    # --- PHASE 1 : COMPARAISON DES INITIALISATIONS ---
    print("--- 1. ÉVALUATION DES SOLUTIONS INITIALES ---")
    
    # Génération Aléatoire (Force Brute)
    sol_aleatoire = random_solution_multi(reseau_c101)
    score_aleatoire = reseau_c101.calculate_objective(sol_aleatoire)
    print(f"[Init] Départ Aléatoire     : {score_aleatoire:.2f} ({len(sol_aleatoire)} camions)")
    
    # Génération Intelligente (Plus Proche Voisin)
    sol_intelligente = nearest_neighbor_solution_multi(reseau_c101)
    score_intelligent = reseau_c101.calculate_objective(sol_intelligente)
    print(f"[Init] Plus Proche Voisin   : {score_intelligent:.2f} ({len(sol_intelligente)} camions)")
    
    print("\n--- 2. L'HYBRIDATION (Amélioration par Métaheuristiques) ---")
    
    # On part du "Plus Proche Voisin" pour gagner un temps précieux
    base_solution = sol_intelligente 
    
    # A. Hill Climbing dégrossit le travail
    print("Lancement du Hill Climbing (Méthode FAST Relocate)...")
    sol_hc = hill_climbing_multi(base_solution, reseau_c101, methode_voisinage="relocate")
    score_hc = reseau_c101.calculate_objective(sol_hc)

    print("Lancement du VND (Relocate -> Swap -> 2-opt)...")
    sol_vnd = vnd_multi(base_solution, reseau_c101)
    score_vnd = reseau_c101.calculate_objective(sol_vnd)

    # B. Tabou prend le relais pour éviter les minimums locaux
    print("Lancement de la Recherche Tabou (200 itérations)...")
    sol_tabu = recherche_tabou_multi(sol_hc, reseau_c101, taille_tabou=20, iter_max=2000, methode_voisinage="swap")
    score_tabu = reseau_c101.calculate_objective(sol_tabu)

    # C. Recuit Simulé (qui part aussi de l'optimum trouvé par HC pour comparer avec Tabou)
    print("Lancement du Recuit Simulé (200 itérations)...")
    sol_recuit = recuit_simule_multi(sol_hc, reseau_c101, temp_initiale=100.0, taux_refroidissement=0.95, iter_max=200, methode_voisinage="relocate")
    score_recuit = reseau_c101.calculate_objective(sol_recuit)

    # --- PHASE 3 : RÉSULTATS GLOBAUX ---
    print("\n=================== RÉSULTATS FINAUX ===================")
    print(f"Optimum mathématique (Ground Truth) : {score_optimal:.2f} ({nb_camions_optimal} camions)")
    print("-" * 56)
    
    # Fonction locale pour afficher le gap proprement
    def print_result(nom, score, nb_cam, optimal):
        gap = 100 * (score - optimal) / optimal
        print(f"{nom:<20} : {score:>8.2f} ({nb_cam:>2} camions) | Gap: {gap:>6.2f} %")

    print_result("1. Aléatoire", score_aleatoire, len(sol_aleatoire), score_optimal)
    print_result("2. Proche Voisin", score_intelligent, len(sol_intelligente), score_optimal)
    print_result("3. Hill Climbing", score_hc, len(sol_hc), score_optimal)
    print_result("4. VND", score_vnd, len(sol_vnd), score_optimal)
    print_result("5. Recherche Tabou", score_tabu, len(sol_tabu), score_optimal)
    print_result("6. Recuit Simulé", score_recuit, len(sol_recuit), score_optimal)
    print("=========================================================\n")

except Exception as e:
    print(f"\n❌ Erreur durant l'exécution : {e}")

 BENCHMARK ADEME : Instance data/C208.txt

--- 1. ÉVALUATION DES SOLUTIONS INITIALES ---
[Init] Départ Aléatoire     : 3662.94 (13 camions)
[Init] Plus Proche Voisin   : 644.95 (3 camions)

--- 2. L'HYBRIDATION (Amélioration par Métaheuristiques) ---
Lancement du Hill Climbing (Méthode FAST Relocate)...
Lancement du VND (Relocate -> Swap -> 2-opt)...
Lancement de la Recherche Tabou (200 itérations)...
Lancement du Recuit Simulé (200 itérations)...

=================== RÉSULTATS FINAUX ===================
Optimum mathématique (Ground Truth) : 585.80 (3 camions)
--------------------------------------------------------
1. Aléatoire         :  3662.94 (13 camions) | Gap: 525.29 %
2. Proche Voisin     :   644.95 ( 3 camions) | Gap:  10.10 %
3. Hill Climbing     :   644.95 ( 3 camions) | Gap:  10.10 %
4. VND               :   618.99 ( 3 camions) | Gap:   5.66 %
5. Recherche Tabou   :   618.05 ( 3 camions) | Gap:   5.50 %
6. Recuit Simulé     :   644.95 ( 3 camions) | Gap:  10.10 %

